# **DATA BALANCING NOTEBOOK**

## **0. Library import**

This section imports the libraries and modules needed for data balancing. This includes data processing libraries such as pandas, sklearn for data splitting, and custom modules for feature engineering and data balancing.

In [1]:
import os
import sys

# Add the root path into the python path
root_path = os.path.abspath(os.path.join(".."))
if not root_path in sys.path:
    sys.path.insert(0, root_path)

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from src.utils import save_data
from src.config import BRFSS_CLEANED_FILE_PATH, LOG_DIR, TRAINING_DIR, \
    RANDOM_OVER_SAMPLING_DATA_FILE_PATH, SMOTE_DATA_FILE_PATH, ADASYN_DATA_FILE_PATH, \
    RANDOM_UNDER_SAMPLING_DATA_FILE_PATH, TOMEK_LINKS_DATA_FILE_PATH, EDITED_NEIGHBORS_DATA_FILE_PATH, \
    SMOTE_TOMEK_LINKS_DATA_FILE_PATH, SMOTE_ENN_DATA_FILE_PATH, ADASYN_TOMEK_DATA_FILE_PATH, \
    TESTING_DATA_FILE_PATH, TRAIN_SIZE, N_JOBS, OVER_SAMPLING_STRATEGY, UNDER_SAMPLING_STRATEGY
from src.features import DiabetesFeatureEngineering
from src.balancing import OverSamplingBalancer, UnderSamplingBalancer, HybridSamplingBalancer

## **1. Load data**

Load preprocessed data from a CSV file. This data has undergone filtering and cleaning from previous steps, containing information about health indicators and the Diabetes target variable with three classes: No diabetes (0), Pre-diabetes (1), and Diabetes (2).

In [3]:
# Load preprocessed BRFSS dataset for data balancing
# Contains imbalanced classes: No diabetes (81.8%), Pre-diabetes (2.4%), Diabetes (15.8%)
df = pd.read_csv(filepath_or_buffer=BRFSS_CLEANED_FILE_PATH)
df.head()

,Diabetes,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,HvyAlcoholConsump,...,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income,Year
0,2.0,1.0,1.0,1.0,27.0,0.0,0.0,0.0,1.0,0.0,...,0.0,2.0,0.0,0.0,1.0,0.0,11.0,6.0,6.0,2017
1,0.0,1.0,0.0,1.0,29.0,0.0,0.0,0.0,1.0,0.0,...,0.0,2.0,0.0,0.0,0.0,1.0,10.0,6.0,8.0,2017
2,0.0,0.0,0.0,1.0,23.0,1.0,0.0,0.0,0.0,0.0,...,0.0,4.0,0.0,14.0,0.0,0.0,10.0,2.0,2.0,2017
3,0.0,1.0,0.0,1.0,27.0,1.0,0.0,1.0,1.0,0.0,...,0.0,3.0,0.0,6.0,0.0,1.0,12.0,4.0,4.0,2017
4,0.0,0.0,0.0,1.0,28.0,0.0,0.0,0.0,0.0,0.0,...,0.0,3.0,0.0,0.0,0.0,1.0,10.0,5.0,8.0,2017


## **2. Feature engineering**

Apply feature engineering techniques to create new features from the original data to improve model performance. This process includes creating composite metrics such as Health Score, Risk Score, and categorical variables such as BMI Category, Age Group. The goal is to create highly informative features that help the machine learning model recognize important patterns in the data.

Generate composite indices from multiple related variables:

- **Health Score**: Composite health index based on GenHlth, MentHlth, PhysHlth
- **Risk Score**: Risk score based on HighBP, HighChol, HeartDiseaseorAttack, Stroke
- **Lifestyle Score**: Lifestyle score based on PhysActivity and negative factors such as HvyAlcoholConsump, Smoker
- **Cardio Risk**: Cardiovascular risk based on HighBP, HighChol and BMI obesity

Convert continuous variables into meaningful categories:

- **BMI Category**: BMI classification according to WHO standards (Underweight, Normal, Pre-obesity, Obesity class I-III)
- **Age Group**: Age grouping into meaningful ranges (Young, Middle-aged, Senior, Elderly)


In [4]:
# Apply a complete feature engineering pipeline:
# - Remove duplicates and handle outliers
# - Create composite scores (Health, Risk, Lifestyle, Cardio)
# - Feature selection based on a correlation threshold (>0.1)
diabetes_feature_engineering = DiabetesFeatureEngineering(log_file=f"{LOG_DIR}/4_feature_engineering_pipeline.log")
processed_df, encoders, selected_features = diabetes_feature_engineering.process_all(df=df)
processed_df.shape

2025-09-17 16:05:51,155 - [src.features] - INFO - DiabetesFeatureEngineering initialized successfully
2025-09-17 16:05:51,160 - [src.features] - INFO - ============================================================
2025-09-17 16:05:51,162 - [src.features] - INFO - STARTING COMPLETE FEATURE ENGINEERING PIPELINE
2025-09-17 16:05:51,165 - [src.features] - INFO - ============================================================
2025-09-17 16:05:51,168 - [src.features] - INFO - Initial dataset shape: (787602, 21)
2025-09-17 16:05:51,170 - [src.features] - INFO - Starting null values removal process...
2025-09-17 16:05:51,234 - [src.features] - INFO - No null values found in the dataset
2025-09-17 16:05:51,377 - [src.features] - INFO - Null values removal completed. Removed 0 rows (0.00%)
2025-09-17 16:05:51,378 - [src.features] - INFO - Dataset shape: 787602 -> 787602 rows
2025-09-17 16:05:51,380 - [src.features] - INFO - After null removal: (787602, 21)
2025-09-17 16:05:51,382 - [src.features] - 

(702516, 15)

In [5]:
# Save selected features
save_data(
    path=f"{TRAINING_DIR}/selected_features.pkl", 
    data=selected_features
)
# Save encoders
save_data(
    path=f"{TRAINING_DIR}/encoders.pkl", 
    data=encoders
)

In [6]:
# Display the original class imbalance before balancing
# Class 0 (No diabetes): ~82% | Class 1 (Pre-diabetes): ~2% | Class 2 (Diabetes):
processed_df["Diabetes"].value_counts()

Diabetes
0.0    574442
2.0    111184
1.0     16890
Name: count, dtype: int64

In [7]:
# Get features and target
X = processed_df.drop(columns=["Diabetes"])
y = processed_df["Diabetes"]

In [8]:
# Split training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    train_size=TRAIN_SIZE, 
    random_state=42, 
    stratify=y
)

In [9]:
# Export testing data
save_data(
    path=TESTING_DATA_FILE_PATH,
    data={
        "X": X_test,
        "y": y_test
    }
)

## **3. Data balancing**

Addressing the data imbalance issue in the BRFSS dataset, where the "No diabetes" class is the majority (459,553 samples) compared to "Pre-diabetes" (13,512 samples) and "Diabetes" (88,947 samples). Data balance is extremely important to ensure that the model is not biased towards the majority class and can predict accurately for all classes.

### **3.1 Oversampling**

In [10]:
# Initialize OverSamplingBalancer
over_sampling_balancer = OverSamplingBalancer(log_file=f"{LOG_DIR}/4_over_sampling_balancer.log")

2025-09-17 16:06:01,668 - [src.balancing] - INFO - OverSamplingBalancer initialized successfully


#### **3.1.1 Random Oversampling**

In [11]:
# Apply Random Oversampling method with sampling strategy {1.0: 50000,2.0: 100000} on training data
ros_X_train, ros_y_train = over_sampling_balancer.apply_random_oversampling(
    X=X_train, 
    y=y_train, 
    sampling_strategy=OVER_SAMPLING_STRATEGY
)

2025-09-17 16:06:01,694 - [src.balancing] - INFO - Starting Random Over Sampling process...
2025-09-17 16:06:01,715 - [src.balancing] - INFO - Original class distribution:
2025-09-17 16:06:01,717 - [src.balancing] - INFO -   - Class 0.0: 459553 samples
2025-09-17 16:06:01,718 - [src.balancing] - INFO -   - Class 1.0: 13512 samples
2025-09-17 16:06:01,724 - [src.balancing] - INFO -   - Class 2.0: 88947 samples
2025-09-17 16:06:02,004 - [src.balancing] - INFO - New class distribution after Random Over Sampling:
2025-09-17 16:06:02,007 - [src.balancing] - INFO -   - Class 0.0: 459553 samples (+0)
2025-09-17 16:06:02,010 - [src.balancing] - INFO -   - Class 1.0: 50000 samples (+36488)
2025-09-17 16:06:02,011 - [src.balancing] - INFO -   - Class 2.0: 150000 samples (+61053)
2025-09-17 16:06:02,013 - [src.balancing] - INFO - Dataset size: 562012 -> 659553 samples (+97541)
2025-09-17 16:06:02,015 - [src.balancing] - INFO - Random Over Sampling completed successfully


In [12]:
# Verify balanced distribution after resampling
ros_y_train.value_counts()

Diabetes
0.0    459553
2.0    150000
1.0     50000
Name: count, dtype: int64

In [13]:
# Save Random Oversampling balanced dataset for model training
save_data(
    path=RANDOM_OVER_SAMPLING_DATA_FILE_PATH,
    data={
        "X": ros_X_train,
        "y": ros_y_train,
    }
)

#### **3.1.2 SMOTE**

In [14]:
# Apply SMOTE method with sampling strategy {1.0: 50000,2.0: 100000} on training data
smote_X_train, smote_y_train = over_sampling_balancer.apply_smote(
    X=X_train, 
    y=y_train, 
    sampling_strategy=OVER_SAMPLING_STRATEGY
)

2025-09-17 16:06:02,381 - [src.balancing] - INFO - Starting SMOTE process...
2025-09-17 16:06:02,397 - [src.balancing] - INFO - Original class distribution:
2025-09-17 16:06:02,400 - [src.balancing] - INFO -   - Class 0.0: 459553 samples
2025-09-17 16:06:02,402 - [src.balancing] - INFO -   - Class 1.0: 13512 samples
2025-09-17 16:06:02,404 - [src.balancing] - INFO -   - Class 2.0: 88947 samples
2025-09-17 16:06:23,357 - [src.balancing] - INFO - New class distribution after SMOTE:
2025-09-17 16:06:23,357 - [src.balancing] - INFO -   - Class 0.0: 459553 samples (+0 synthetic)
2025-09-17 16:06:23,358 - [src.balancing] - INFO -   - Class 1.0: 50000 samples (+36488 synthetic)
2025-09-17 16:06:23,359 - [src.balancing] - INFO -   - Class 2.0: 150000 samples (+61053 synthetic)
2025-09-17 16:06:23,361 - [src.balancing] - INFO - Dataset size: 562012 -> 659553 samples (+97541 synthetic)
2025-09-17 16:06:23,363 - [src.balancing] - INFO - SMOTE completed successfully


In [15]:
# Verify balanced distribution after resampling
smote_y_train.value_counts()

Diabetes
0.0    459553
2.0    150000
1.0     50000
Name: count, dtype: int64

In [16]:
# Save SMOTE balanced dataset for model training
save_data(
    path=SMOTE_DATA_FILE_PATH,
    data={
        "X": smote_X_train,
        "y": smote_y_train,
    }
)

#### **3.1.3 ADASYN**

In [17]:
# Apply ADASYN method with sampling strategy {1.0: 50000,2.0: 100000} on training data
adasyn_X_train, adasyn_y_train = over_sampling_balancer.apply_adasyn(
    X=X_train, 
    y=y_train, 
    sampling_strategy=OVER_SAMPLING_STRATEGY
)

2025-09-17 16:06:23,704 - [src.balancing] - INFO - Starting ADASYN over-sampling process...
2025-09-17 16:06:23,720 - [src.balancing] - INFO - Original class distribution:
2025-09-17 16:06:23,723 - [src.balancing] - INFO -  - Class 0.0: 459553 samples
2025-09-17 16:06:23,724 - [src.balancing] - INFO -  - Class 1.0: 13512 samples
2025-09-17 16:06:23,726 - [src.balancing] - INFO -  - Class 2.0: 88947 samples
2025-09-17 16:07:02,952 - [src.balancing] - INFO - New class distribution after ADASYN:
2025-09-17 16:07:02,953 - [src.balancing] - INFO -  - Class 0.0: 459553 samples (+0 synthetic)
2025-09-17 16:07:02,953 - [src.balancing] - INFO -  - Class 1.0: 50523 samples (+37011 synthetic)
2025-09-17 16:07:02,954 - [src.balancing] - INFO -  - Class 2.0: 147562 samples (+58615 synthetic)
2025-09-17 16:07:02,954 - [src.balancing] - INFO - Dataset size: 562012 -> 657638 samples (+95626 synthetic)
2025-09-17 16:07:02,955 - [src.balancing] - INFO - ADASYN over-sampling completed successfully


In [18]:
# Verify balanced distribution after resampling
adasyn_y_train.value_counts()

Diabetes
0.0    459553
2.0    147562
1.0     50523
Name: count, dtype: int64

In [19]:
# Save ADASYN balanced dataset for model training
save_data(
    path=ADASYN_DATA_FILE_PATH,
    data={
        "X": adasyn_X_train,
        "y": adasyn_y_train,
    }
)

### **3.2 Undersampling**

The method reduces the number of samples of the majority class to balance with the minority classes, which reduces training time but may lose important information.

In [20]:
# Initialize UnderSamplingBalancer
under_sampling_balancer = UnderSamplingBalancer(log_file=f"{LOG_DIR}/4_under_sampling_balancer.log")

2025-09-17 16:07:03,111 - [src.balancing] - INFO - UnderSamplingBalancer initialized successfully


#### **3.2.1 Random Undersampling**

In [21]:
# Apply Random Undersampling method with {0: 100000, 2: 100000, 1: 16890} on training data
rus_X_train, rus_y_train = under_sampling_balancer.apply_random_undersampling(
    X=X_train, 
    y=y_train,
    sampling_strategy=UNDER_SAMPLING_STRATEGY
)

2025-09-17 16:07:03,119 - [src.balancing] - INFO - Starting Random Under Sampling process...
2025-09-17 16:07:03,125 - [src.balancing] - INFO - Original class distribution:
2025-09-17 16:07:03,126 - [src.balancing] - INFO -   - Class 0.0: 459553 samples
2025-09-17 16:07:03,127 - [src.balancing] - INFO -   - Class 1.0: 13512 samples
2025-09-17 16:07:03,128 - [src.balancing] - INFO -   - Class 2.0: 88947 samples
2025-09-17 16:07:03,272 - [src.balancing] - INFO - New class distribution after Random Under Sampling:
2025-09-17 16:07:03,273 - [src.balancing] - INFO -   - Class 0.0: 100000 samples (-359553)
2025-09-17 16:07:03,275 - [src.balancing] - INFO -   - Class 1.0: 13512 samples (-0)
2025-09-17 16:07:03,278 - [src.balancing] - INFO -   - Class 2.0: 88947 samples (-0)
2025-09-17 16:07:03,280 - [src.balancing] - INFO - Dataset size: 562012 -> 202459 samples (-359553)
2025-09-17 16:07:03,282 - [src.balancing] - INFO - Random Under Sampling completed successfully


In [22]:
# Verify balanced distribution after resampling
rus_y_train.value_counts()

Diabetes
0.0    100000
2.0     88947
1.0     13512
Name: count, dtype: int64

In [23]:
# Save Random Undersamplinng balanced dataset for model training
save_data(
    path=RANDOM_UNDER_SAMPLING_DATA_FILE_PATH,
    data={
        "X": rus_X_train,
        "y": rus_y_train,
    }
)

#### **3.2.2 TomekLinks**

The under-sampling method is smarter, only discarding the majority class samples that are close to the boundary decision and may cause noise. TomekLinks identifies pairs of samples from different classes that are nearest neighbors to each other and discards the majority class sample in that pair. Results: 444,228 (Class 0), 13,512 (Class 1), 74,901 (Class 2)—retains more information than random undersampling

In [24]:
# Apply Tomek Links method with sampling strategy 'auto' on training data
tomek_X_train, tomek_y_train = under_sampling_balancer.apply_tomek_links(
    X=X_train, 
    y=y_train, 
    n_jobs=N_JOBS
)

2025-09-17 16:07:03,365 - [src.balancing] - INFO - Starting Tomek Links process...
2025-09-17 16:07:03,374 - [src.balancing] - INFO - Original class distribution:
2025-09-17 16:07:03,376 - [src.balancing] - INFO -   - Class 0.0: 459553 samples
2025-09-17 16:07:03,377 - [src.balancing] - INFO -   - Class 1.0: 13512 samples
2025-09-17 16:07:03,378 - [src.balancing] - INFO -   - Class 2.0: 88947 samples
2025-09-17 16:07:50,388 - [src.balancing] - INFO - New class distribution after Tomek Links:
2025-09-17 16:07:50,391 - [src.balancing] - INFO -   - Class 0.0: 450210 samples (-9343 Tomek links)
2025-09-17 16:07:50,392 - [src.balancing] - INFO -   - Class 1.0: 13512 samples (unchanged)
2025-09-17 16:07:50,394 - [src.balancing] - INFO -   - Class 2.0: 79599 samples (-9348 Tomek links)
2025-09-17 16:07:50,395 - [src.balancing] - INFO - Dataset size: 562012 -> 543321 samples (-18691 Tomek links)
2025-09-17 16:07:50,397 - [src.balancing] - INFO - Tomek Links processing completed successfully


In [25]:
# Verify balanced distribution after resampling
tomek_y_train.value_counts()

Diabetes
0.0    450210
2.0     79599
1.0     13512
Name: count, dtype: int64

In [26]:
# Save Tomek Links balanced dataset for model training
save_data(
    path=TOMEK_LINKS_DATA_FILE_PATH,
    data={
        "X": tomek_X_train,
        "y": tomek_y_train,
    }
)

#### **3.2.3 Edited Nearest Neighbors**

In [27]:
# Apply Edited Nearest Neighbors method with sampling strategy 'auto' on training data
enn_X_train, enn_y_train = under_sampling_balancer.apply_edited_nearest_neighbours(
    X=X_train, 
    y=y_train,
    n_jobs=N_JOBS,
)

2025-09-17 16:07:50,694 - [src.balancing] - INFO - Starting Edited Nearest Neighbours (ENN) under-sampling process...
2025-09-17 16:07:50,710 - [src.balancing] - INFO - Original class distribution:
2025-09-17 16:07:50,712 - [src.balancing] - INFO -  - Class 0.0: 459553 samples
2025-09-17 16:07:50,713 - [src.balancing] - INFO -  - Class 1.0: 13512 samples
2025-09-17 16:07:50,715 - [src.balancing] - INFO -  - Class 2.0: 88947 samples
2025-09-17 16:09:21,559 - [src.balancing] - INFO - New class distribution after ENN:
2025-09-17 16:09:21,562 - [src.balancing] - INFO -  - Class 0.0: 353329 samples (-106224 removed)
2025-09-17 16:09:21,563 - [src.balancing] - INFO -  - Class 1.0: 13512 samples (-0 removed)
2025-09-17 16:09:21,564 - [src.balancing] - INFO -  - Class 2.0: 28181 samples (-60766 removed)
2025-09-17 16:09:21,566 - [src.balancing] - INFO - Dataset size: 562012 -> 395022 samples (-166990)
2025-09-17 16:09:21,567 - [src.balancing] - INFO - ENN under-sampling completed successfully


In [28]:
# Verify balanced distribution after resampling
enn_y_train.value_counts()

Diabetes
0.0    353329
2.0     28181
1.0     13512
Name: count, dtype: int64

In [29]:
# Save Edited Nearest Neighbors balanced dataset for model training
save_data(
    path=EDITED_NEIGHBORS_DATA_FILE_PATH,
    data={
        "X": enn_X_train,
        "y": enn_y_train,
    }
)

### **3.3 Hybrid**

Combine both over-sampling and under-sampling to get the best of both worlds, creating a balanced dataset with the highest quality.

In [30]:
# Initialize HybridSamplingBalancer
hybrid_sampling_balancer = HybridSamplingBalancer(log_file=f"{LOG_DIR}/4_hybrid_sampling_balancer.log")

2025-09-17 16:09:21,953 - [src.balancing] - INFO - HybridSamplingBalancer initialized successfully


#### **3.3.1 SMOTE + Tomek Links**

Combining SMOTE and TomekLinks: first apply SMOTE to upsample the minority class, then use TomekLinks to clean the boundary and remove noisy samples. This method produces a balanced dataset with high quality, reduces noise, improves classification ability at the boundary. Results: 564,423 (Class 0), 149,036 (Class 1), 90,450 (Class 2).


In [31]:
# Apply SMOTE + Tomek Links method with sampling strategy 'auto' on training data
smote_tomek_X_train, smote_tomek_y_train = hybrid_sampling_balancer.apply_smote_tomek(
    X=X_train, 
    y=y_train,
    n_jobs=N_JOBS,
)

2025-09-17 16:09:21,994 - [src.balancing] - INFO - Starting SMOTETomek hybrid sampling process...
2025-09-17 16:09:22,014 - [src.balancing] - INFO - Original class distribution:
2025-09-17 16:09:22,016 - [src.balancing] - INFO -   - Class 0.0: 459553 samples
2025-09-17 16:09:22,018 - [src.balancing] - INFO -   - Class 1.0: 13512 samples
2025-09-17 16:09:22,020 - [src.balancing] - INFO -   - Class 2.0: 88947 samples
2025-09-17 16:11:36,249 - [src.balancing] - INFO - New class distribution after SMOTETomek:
2025-09-17 16:11:36,250 - [src.balancing] - INFO -   - Class 0.0: 456890 samples (-2663 net)
2025-09-17 16:11:36,251 - [src.balancing] - INFO -   - Class 1.0: 457368 samples (+443856 net)
2025-09-17 16:11:36,251 - [src.balancing] - INFO -   - Class 2.0: 455155 samples (+366208 net)
2025-09-17 16:11:36,253 - [src.balancing] - INFO - Dataset size: 562012 -> 1369413 samples (+807401)
2025-09-17 16:11:36,256 - [src.balancing] - INFO - SMOTETomek processing completed successfully


In [32]:
# Verify balanced distribution after resampling
smote_tomek_y_train.value_counts()

Diabetes
1.0    457368
0.0    456890
2.0    455155
Name: count, dtype: int64

In [33]:
# # Save SMOTE + Tomek Links balanced dataset for model training
save_data(
    path=SMOTE_TOMEK_LINKS_DATA_FILE_PATH,
    data={
        "X": smote_tomek_X_train,
        "y": smote_tomek_y_train,
    }
)

#### **3.3.2 SMOTE + ENN**

In [34]:
# Apply SMOTE + ENN method with sampling strategy {1.0: 50000, 2.0: 100000} on training data
smoteen_X_train, smoteen_y_train = hybrid_sampling_balancer.apply_smote_enn(
    X=X_train, 
    y=y_train, 
    sampling_strategy=OVER_SAMPLING_STRATEGY, 
    n_jobs=N_JOBS
)

2025-09-17 16:11:36,762 - [src.balancing] - INFO - Starting SMOTEENN hybrid sampling process...
2025-09-17 16:11:36,781 - [src.balancing] - INFO - Original class distribution:
2025-09-17 16:11:36,782 - [src.balancing] - INFO -   - Class 0.0: 459553 samples
2025-09-17 16:11:36,784 - [src.balancing] - INFO -   - Class 1.0: 13512 samples
2025-09-17 16:11:36,785 - [src.balancing] - INFO -   - Class 2.0: 88947 samples
2025-09-17 16:13:22,852 - [src.balancing] - INFO - New class distribution after SMOTEENN:
2025-09-17 16:13:22,854 - [src.balancing] - INFO -   - Class 0.0: 338617 samples (-120936 net)
2025-09-17 16:13:22,855 - [src.balancing] - INFO -   - Class 1.0: 22134 samples (+8622 net)
2025-09-17 16:13:22,857 - [src.balancing] - INFO -   - Class 2.0: 66625 samples (-22322 net)
2025-09-17 16:13:22,858 - [src.balancing] - INFO - Dataset size: 562012 -> 427376 samples (-134636)
2025-09-17 16:13:22,860 - [src.balancing] - INFO - SMOTEENN processing completed successfully


In [35]:
# Verify balanced distribution after resampling
smoteen_y_train.value_counts()

Diabetes
0.0    338617
2.0     66625
1.0     22134
Name: count, dtype: int64

In [36]:
# Save SMOTE + ENN balanced dataset for model training
save_data(
    path=SMOTE_ENN_DATA_FILE_PATH,
    data={
        "X": smoteen_X_train,
        "y": smoteen_y_train,
    }
)

#### **3.3.3 ADASYN + Tomek Links**


In [37]:
# Apply ADASYN + Tomek Links method with sampling strategy {1.0: 50000,2.0: 100000} method on training data
adasyn_tomek_X_train, adasyn_tomek_y_train = hybrid_sampling_balancer.apply_adasyn_tomek(
    X=X_train, 
    y=y_train, 
    sampling_strategy=OVER_SAMPLING_STRATEGY,
    n_jobs=N_JOBS,
)

2025-09-17 16:13:23,062 - [src.balancing] - INFO - Starting ADASYN + TomekLinks hybrid sampling process...
2025-09-17 16:13:23,078 - [src.balancing] - INFO - Original class distribution:
2025-09-17 16:13:23,080 - [src.balancing] - INFO -   - Class 0.0: 459553 samples
2025-09-17 16:13:23,081 - [src.balancing] - INFO -   - Class 1.0: 13512 samples
2025-09-17 16:13:23,082 - [src.balancing] - INFO -   - Class 2.0: 88947 samples
2025-09-17 16:14:58,334 - [src.balancing] - INFO - New class distribution after ADASYN + TomekLinks:
2025-09-17 16:14:58,337 - [src.balancing] - INFO -   - Class 0.0: 454843 samples (-4710 net)
2025-09-17 16:14:58,339 - [src.balancing] - INFO -   - Class 1.0: 50523 samples (+37011 net)
2025-09-17 16:14:58,341 - [src.balancing] - INFO -   - Class 2.0: 142923 samples (+53976 net)
2025-09-17 16:14:58,342 - [src.balancing] - INFO - Dataset size: 562012 -> 648289 samples (+86277)
2025-09-17 16:14:58,343 - [src.balancing] - INFO - ADASYN + TomekLinks processing completed 

In [38]:
# Verify balanced distribution after resampling
adasyn_tomek_y_train.value_counts()

Diabetes
0.0    454843
2.0    142923
1.0     50523
Name: count, dtype: int64

In [39]:
# Save ADASYN + Tomek Links balanced dataset for model training
save_data(
    path=ADASYN_TOMEK_DATA_FILE_PATH,
    data={
        "X": adasyn_tomek_X_train,
        "y": adasyn_tomek_y_train,
    }
)

### **3.4 Comparison of results of methods**

The summary table shows the clear differences between the methods:
- **Over-sampling methods** produce the largest dataset, suitable when all information needs to be kept
- **Under-sampling methods** produce the smallest dataset, suitable when computational resources are limited
- **Hybrid methods** balance size and quality, often giving the best results in practice

The choice of the appropriate method depends on the specific characteristics of the data, the available computational resources, and the model performance requirements.

| Method             | Class 0     | Class 1     | Class 2    | Total Samples  |
|:-------------------|:------------|:------------|:-----------|:---------------|
| Original           | 459,553     | 13,512      | 88,947     | 562,012        |
| Random Oversample  | 574,477     | 150,000     | 100,000    | 824,477        |
| SMOTE              | 574,477     | 150,000     | 100,000    | 824,477        |
| Random Undersample | 13,512      | 13,512      | 13,512     | 40,536         |
| TomekLinks         | 444,228     | 13,512      | 74,901     | 532,641        |
| SMOTETomek         | 564,423     | 149,036     | 90,450     | 803,909        |
| SMOTEEN            | 405,203     | 106,148     | 8,988      | 520,339        |